# Pórtico 253: construya el cálculo

Mismo planteamiento de la lámina253. Las matrices y resultados numéricos son desarrollo de cálculo; la lámina no aporta la solución.

Primero resuelva a mano. Complete cada `None` con una expresión o matriz. Ejecute una celda cada vez. Los `assert` comprueban el procedimiento, no asignan calificaciones. Una celda incompleta se detiene con una indicación. Python es complementario.

Use kN, m y rad, excepto donde se indique una conversión. +X a la derecha, +Y arriba, giros y momentos antihorarios. $S_{ij}>0$ indica tensión. $F_n$ es carga libre; $F_a$ es acción neta en apoyos. La reacción es $F_a$ menos la carga aplicada allí. $w>0$ actúa hacia abajo en la viga horizontal.

Referencia: https://view.genially.com/684f80b53c7df3efd8dc19f4


In [ ]:
import numpy as np
np.set_printoptions(precision=7, suppress=True)
def complete(value, shape, name):
    assert value is not None, f"Complete {name} antes de ejecutar la comprobación."
    out=np.asarray(value,dtype=float)
    assert out.shape==shape, f"Revise dimensiones de {name}: se espera {shape}."
    assert np.isfinite(out).all(), f"Revise valores finitos de {name}."
    return out


## 1. Calcule propiedades y matrices locales

Pórtico253: nudos1=(0,3),2=(6,3),3=(0,0),4=(6,0)m. Columnas3→1,4→2 y viga1→2. E=19×10⁶kN/m². VigaA=.15m²,I=.003125m⁴; columnasA=.09m²,I=.000675m⁴. Complete la función6×6.


In [ ]:
# Actividad P1
E=19e6
xy=np.array([[0.,3.],[6.,3.],[0.,0.],[6.,0.]])
miembros=[(2,0,.09,.000675),(0,1,.15,.003125),(3,1,.09,.000675)]
def local(EA,EI,L):
    return None


In [ ]:
# Compruebe el paso antes de continuar.
kb=complete(local(E*.15,E*.003125,6),(6,6),'Kbar de viga')
assert np.isclose(kb[0,0],475000), 'Revise AE/L y unidades.'
assert np.isclose(kb[2,2],39583.3333333333), 'Revise 4EI/L.'
assert np.allclose(kb,kb.T), 'Revise simetría.'
print(kb)


## 2. Transforme y ensamble

Complete la transformación6×6: R=[[c,s,0],[-s,c,0],[0,0,1]], T=diag(R,R). Complete K de12×12 sumando los tres aportes. Orden [u1,v1,θ1,...,u4,v4,θ4].


In [ ]:
# Actividad P2
def transformar(c,s):
    return None
elementos=[]
K=None


In [ ]:
# Compruebe el paso antes de continuar.
K=complete(K,(12,12),'K')
assert len(elementos)==3, 'Guarde los tres aportes para recuperar fuerzas.'
assert np.allclose(K,K.T), 'Revise ensamblaje.'
assert np.isclose(K[0,0],480700), 'En u1 se suman la flexión de columna y el axial de viga.'
print(K[:6,:6])


## 3. Aplique w y resuelva los movimientos

w=25 kN/m hacia abajo solo en la viga1→2. Su F̄^F=[0,wL/2,wL²/12,0,wL/2,−wL²/12]. Las columnas no tienen cargas intermedias. n=[u1,v1,θ1,u2,v2,θ2], bases inmóviles. Fn=0. Además de reemplazar None, ensamble FF después de inicializarlo con ceros: transforme F̄^F y súmelo en los códigos de la viga.


In [ ]:
# Actividad P3
w=25.
FFbar_viga=None
FF=np.zeros(12)
n=np.arange(6);a=np.arange(6,12)
Knn=None
dn=None


In [ ]:
# Compruebe el paso antes de continuar.
FFbar_viga=complete(FFbar_viga,(6,),'FFbar_viga');Knn=complete(Knn,(6,6),'Knn');dn=complete(dn,(6,),'dn')
assert np.allclose(FFbar_viga,[0,75,75,0,75,-75]), 'Revise signos o kN·m.'
assert np.allclose(Knn@dn,-FF[n]), 'Sustituya en el sistema reducido.'
assert np.allclose(dn, [.00001822548,-.00013157895,-.00203720337,-.00001822548,-.00013157895,.00203720337],atol=1e-10), 'Revise unidades y simetría.'
print(dn)


## 4. Recupere apoyos y acciones locales

Después de inicializar delta, asigne delta[n]=dn. Calcule Fa y los tres vectores F̄=F̄^F+K̄Tδ. Aquí no existen cargas aplicadas directamente en las bases, por lo que Fa coincide con las reacciones.


In [ ]:
# Actividad P4
delta=np.zeros(12)
Fa=None
acciones=None


In [ ]:
# Compruebe el paso antes de continuar.
Fa=complete(Fa,(6,),'Fa');acciones=complete(acciones,(3,6),'acciones')
assert np.allclose(Fa,[17.314203632,75,-17.262261021,-17.314203632,75,17.262261021],atol=1e-7), 'Revise transformación de columnas y filas de apoyo.'
assert np.allclose(acciones[1],[17.314203632,75,34.680349875,-17.314203632,75,-34.680349875],atol=1e-7), 'Incluya F^F al recuperar la viga.'
assert np.allclose(acciones[0],[75,-17.314203632,-17.262261021,-75,17.314203632,-34.680349875],atol=1e-7), 'Revise la columna 3→1 en sus ejes locales.'
assert np.allclose(acciones[2],[75,17.314203632,17.262261021,-75,-17.314203632,34.680349875],atol=1e-7), 'Revise la columna 4→2 en sus ejes locales.'
for k,el in enumerate(elementos):
    carga_tramo=w if k==1 else 0.
    assert np.isclose(acciones[k,1]+acciones[k,4]-carga_tramo*el['L'],0), 'No cierra el equilibrio transversal del elemento.'
    assert np.isclose(acciones[k,2]+acciones[k,5]+acciones[k,4]*el['L']-carga_tramo*el['L']**2/2,0), 'No cierra el momento del elemento respecto a i.'
assert np.isclose(Fa[1]+Fa[4]-w*6,0), 'No cierra el equilibrio vertical.'
print('Fa:',Fa);print('Acciones locales por fila 31,12,42:',acciones)


## Explique el resultado

Escriba el equilibrio del nudo1 sumando las acciones de columna y viga en ejes globales. Distinga los momentos de extremo aplicados al elemento del momento interno sagante. Para la viga, M(x)=−34.680349875+75x−12.5x² kN·m. Obtenga su máximo y explique la compresión axial.

Si una prueba falla, compare la primera operación distinta, no solo la respuesta final. Consulte `soluciones_laboratorios.md` después de intentar el paso. El notebook resuelto del mismo ejemplo sirve como referencia adicional.
